# Act 1 — The machines

Compute Engine is the IaaS layer of GCP — virtual machines you can shape, attach disks to, place in a network, and run your code on. Everything higher up (Cloud Run, GKE, Cloud Functions) eventually rents Compute Engine capacity under the hood, so understanding it is useful even if you'll mostly live in PaaS later.

The core decisions are: which CPU/memory shape, which boot image, where it runs, and how isolated the underlying hardware needs to be. We'll work through those, then move to how you pay for the resulting VMs.

## Machine families

GCP groups VM shapes into **machine families** by workload profile. You pick a family, then a **machine type** within it (vCPU + memory). Five families cover most of what you'll see in practice:

| Family | Profile | Examples | When to reach for it |
|---|---|---|---|
| **E2** | Cost-optimised, shared CPU | `e2-standard-4`, `e2-medium` | Dev/test, low-traffic web, anything where price beats performance |
| **N2 / N4** | General-purpose Intel | `n2-standard-8`, `n4-highmem-16` | Default for most production workloads |
| **C3 / C3D / C4** | Compute-optimised | `c3-standard-22`, `c4-highcpu-48` | CPU-heavy: web serving at scale, gaming servers, batch CPU |
| **M3 / M2** | Memory-optimised | `m3-megamem-128` | In-memory databases (SAP HANA), large Redis/Memcached |
| **A3 / A2 / G2** | Accelerator (GPU/TPU) | `a3-highgpu-8g`, `g2-standard-8` | ML training, inference, rendering |
| **T2D / T2A** | Tau — high-cost-performance | `t2d-standard-16` | Scale-out workloads where per-core price matters more than peak per-core performance |

**Predefined vs custom types.** Most teams use predefined types (`n2-standard-8`). Custom types (`n2-custom-8-16384` for 8 vCPU + 16 GiB) exist when no predefined shape fits — useful for licensed software with awkward CPU/RAM ratios.

**Sizing rule of thumb.** Start one tier smaller than you think and let monitoring tell you to grow. Right-sizing is a notebook-14 conversation, but the first instinct "surely I need n2-standard-32" is wrong about 80% of the time.

## Images — what the VM boots from

A VM boots from an **image**, which becomes the root of its boot disk. Three flavours:

- **Public images** — maintained by Google or partners. Debian, Ubuntu, RHEL, Windows Server, Container-Optimized OS (COS, the GKE/Cloud Run host OS). Always patched to the latest snapshot when you pick the family name (`debian-12`).
- **Custom images** — created from one of your own VM boot disks (`gcloud compute images create … --source-disk …`). The standard pattern: bake configuration in with Packer, push the resulting image to your project, point an instance template at it.
- **Image families** — a moving alias pointing at the latest image in a family (e.g. `acme-webapp-prod` always resolves to whichever image you tagged most recently). Used heavily by MIGs so rolling updates can point at the family and pick up new images automatically.

**Container-Optimized OS** is the one to know by name. It's a minimal, signed, read-only-root host OS built only to run containers. GKE nodes use it. Cloud Run uses it. If you're running Docker on a long-lived VM, COS is usually the better choice over a general-purpose Linux.

## Sole-tenant nodes

By default, your VM shares physical hardware with other tenants — that's normal cloud multi-tenancy, and the hypervisor isolates you.

**Sole-tenant nodes** rent you an entire physical host. Only your VMs run on it. Two reasons to care:

- **Licensing** — Bring-Your-Own-License Windows / Oracle / SQL Server often requires per-physical-socket licensing, which only works if you own the socket.
- **Compliance** — "no shared hardware" is sometimes mandated by regulation or contract.

You pay a premium for the dedicated host on top of the VM-hour cost. For most workloads, this isn't worth it — but when it's required, it's the only way.

# Act 2 — Paying for compute

Four pricing models stack on top of each other. Pick the right combination per workload and the bill drops 30–70% off the on-demand sticker price. Pick badly — or default to on-demand for everything — and you're leaving money on the table that finance will eventually notice.

The four are: **on-demand**, **Sustained Use Discounts**, **Committed Use Discounts**, and **Spot**. SUD is unique to GCP and the most often missed.

## Four pricing models

| Model | What you commit to | Discount | When to use |
|---|---|---|---|
| **On-demand** | Nothing | 0% | Bursty, unpredictable, short-lived |
| **Sustained Use Discount (SUD)** | Nothing (automatic) | Up to ~30% | Any VM that runs >25% of a month — applied automatically |
| **Committed Use Discount (CUD)** | 1 or 3 years of spend | Up to ~57% (3-yr resource-based) | Predictable baseline you'll definitely consume |
| **Spot VMs** | Nothing, but Google can preempt you | 60–91% | Fault-tolerant batch, stateless workers, build agents |

**Sustained Use Discount — the GCP-unique one.** SUDs apply *automatically* with no commitment. The longer a VM runs in a given month, the bigger the discount. No reservations to manage, no expiry to track. AWS's nearest equivalent is the (now somewhat reduced) Savings Plans baseline; Azure has nothing directly comparable. If you have a steady-state workload running 24/7, you're already getting roughly 30% off without doing anything.

**Committed Use Discounts.** Two kinds:

- **Resource-based CUDs** — commit to a specific machine type in a specific region. Best discount, least flexible.
- **Spend-based CUDs** — commit to dollars-per-hour across a service (Compute Engine, Cloud SQL, others). Lower discount, far more flexible. The right default for most teams.

**Spot VMs.** Replaced the older "preemptible VMs" model. Spot has no fixed 24-hour cap (preemptible did), and prices float by region/family. Same fundamental contract: Google can shut down the VM with ~30 seconds' notice when capacity is needed elsewhere. Always run Spot inside a Managed Instance Group with autohealing so preemption recovers cleanly.

# Act 3 — Instance templates and Managed Instance Groups

One VM is interesting; ten thousand identical VMs that scale up and down by themselves is what you actually want in production. That's what Managed Instance Groups (MIGs) deliver.

The pattern is the same on every cloud — define the shape once, ask the platform to keep N copies alive — but the GCP version has a few details worth knowing. Zonal vs regional MIGs, rolling update modes, and the way autohealing uses health checks all shape how reliably your fleet recovers from things going wrong.

## Instance templates — the immutable spec

An **instance template** is a frozen recipe for a VM: machine type, boot image, attached disks, startup script, network tags, service account, labels. You don't edit a template once it exists — you create a new version and point your MIG at the new one.

That immutability is the whole point. A MIG using template `v3` and another using `v4` are unambiguously different fleets; rollbacks are "point the MIG back at v3 and let it roll forward."

## Managed Instance Groups — zonal vs regional

A **Managed Instance Group (MIG)** keeps a target number of identical VMs alive by creating them from an instance template, watching their health, and replacing them when they break.

Two flavours:

- **Zonal MIG** — instances live in a single zone (e.g. `us-central1-a`). Simpler, but a zone outage takes the whole MIG down.
- **Regional MIG** — instances spread across three or more zones in a region. This is the default for anything you care about being up. A zone outage drops a third of your fleet; the other two thirds keep serving while the MIG works on replacements.

**Always regional in production.** The cost difference is essentially zero — you'd put VMs across zones anyway for HA — and the resilience win is large.

Four features sit on top of every MIG:

1. **Autoscaling** — grow or shrink the group based on CPU, load-balancer requests-per-second, a Cloud Monitoring metric, or a schedule. Define min/max; the autoscaler does the rest.
2. **Autohealing** — attach a health check; instances that fail it are recreated automatically. Without autohealing, a wedged process inside a VM serves errors forever; with it, the VM gets replaced within minutes.
3. **Rolling updates** — change the instance template and the MIG cycles old VMs out and new ones in, with control over batch size and surge.
4. **Instance protection** — mark specific instances as "don't touch" during scale-in or rolling updates. Useful for the one VM holding warm state.

**Unmanaged Instance Groups** are the opposite — a hand-curated bag of VMs that share nothing in common except your decision to group them. They exist for legacy load-balancer backend compatibility. Don't reach for them for anything new.

## Rolling updates — three modes

When you change the template, the MIG can roll the change out three ways:

| Mode | What it does | When to use |
|---|---|---|
| **Proactive** | Replace existing instances immediately, in waves | Standard production rollout |
| **Opportunistic** | Only replace instances when they're naturally recreated (autohealed, scaled, manually deleted) | Long-tail rollouts with no urgency |
| **Canary** | Apply the new template to a fraction of instances first; if happy, promote to all | Risky changes you want to observe before committing |

**Control knobs** worth knowing:

- **`max-surge`** — how many *extra* instances the MIG can spin up beyond the target during the rollout. Set higher for faster rollouts.
- **`max-unavailable`** — how many instances can be down at once. Set to 0 if your fleet is small and you can't afford lost capacity during rollout.
- **`min-ready-sec`** — how long a new instance has to be healthy before it counts as ready. Tune this to your warmup time.

In [ ]:
# A regional MIG with autoscaling and autohealing. Shown as gcloud
# because in practice you'll define this in Terraform — and Terraform's
# google_compute_region_instance_group_manager resource takes nearly
# the same arguments under different names.
#
# gcloud compute instance-templates create acme-web-v3 \
#   --machine-type=n2-standard-4 \
#   --image-family=acme-webapp-prod \
#   --image-project=acme-images \
#   --tags=web,allow-lb-health \
#   --service-account=app-runtime@acme-prod.iam.gserviceaccount.com \
#   --scopes=cloud-platform
#
# gcloud compute health-checks create http acme-web-hc \
#   --port=8080 --request-path=/healthz --check-interval=10s
#
# gcloud compute instance-groups managed create acme-web-mig \
#   --region=us-central1 \
#   --template=acme-web-v3 \
#   --size=6 \
#   --health-check=acme-web-hc \
#   --initial-delay=120
#
# gcloud compute instance-groups managed set-autoscaling acme-web-mig \
#   --region=us-central1 \
#   --min-num-replicas=3 --max-num-replicas=30 \
#   --target-cpu-utilization=0.6

# Act 4 — Availability and what it actually buys you

You've seen the building blocks. The remaining question: given zones, regions, regional MIGs, and autohealing, what does the *availability* picture actually look like in practice? Three things matter — failure scopes, the role of health checks, and a clear-eyed view of which problems Compute Engine HA does and doesn't solve.

## Failure scopes — what HA at each layer protects against

| Failure | Single VM | Zonal MIG | Regional MIG | Multi-region |
|---|---|---|---|---|
| One VM crashes | ❌ Outage | ✅ Autoheal | ✅ Autoheal | ✅ Autoheal |
| One AZ-equivalent (zone) goes down | ❌ Outage | ❌ Outage | ✅ Two zones still up | ✅ Other region up |
| Whole region goes down (rare but happens) | ❌ Outage | ❌ Outage | ❌ Outage | ✅ Other region up |
| Bad rollout takes down the app | ❌ Outage | ❌ Outage | ⚠️ Partial (canary if used) | ⚠️ Partial |

Most production workloads stop at **regional MIG**. The jump to multi-region is significant in cost and complexity (mostly because of state — see notebook 13 on DR strategies), and only the highest-availability requirements justify it.

The last row is the honest one. Compute Engine HA protects against *infrastructure* failures. It does nothing about *bad code you just deployed*. Canary rollouts and good observability are what guard against that — both topics for later notebooks.

## Health checks — used in two different ways

A health check is a periodic probe (HTTP/HTTPS/TCP/SSL/gRPC) against a backend. Two distinct GCP features both use them, and the difference matters:

- **MIG autohealing health checks** decide whether to **recreate** an instance. Mark these conservatively — too-strict checks lead to constant replacement during transient blips.
- **Load balancer health checks** decide whether to **route traffic** to an instance. These can be stricter — you want unhealthy instances out of rotation quickly.

The two often look identical (`/healthz` returning 200), and that's fine — but they live in different resources and are configured separately. Notebook 07 covers LB health checks in depth.

## What carries into later chapters

The Compute Engine building blocks underpin almost everything else. GKE nodes are Compute Engine VMs. Cloud Run runs on Compute Engine under the hood. Cloud SQL is a managed Compute Engine VM with a database engine pre-installed.

Three habits worth carrying forward:

- **Regional MIGs by default.** Single-zone deployment is a teaching aid; production is regional.
- **Spot for stateless workers, never for state.** Build agents, batch jobs, render farms — Spot is right. Anything holding session state, in-memory caches, or a database — never Spot.
- **Sustained Use Discounts apply automatically; Committed Use Discounts need to be bought.** Steady-state spend without CUDs is the most common cost-optimization win on the table for a team that's been on GCP for a year and hasn't looked at billing yet.

Notebook 04 leaves Compute Engine and steps up the stack to Cloud Run, Cloud Functions, and GKE — where the platform manages more, you manage less, and the trade-off (cold starts, container constraints, less hardware choice) starts to bite.